# Text Summarization using TF-IDF

In [147]:
import polars as pl
import numpy as np

import nltk
import textwrap
from nltk import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("punkt", "../datasets/nltk/")
nltk.download("stopwords", "../datasets/nltk/")

nltk.data.path.append("../datasets/nltk/")

[nltk_data] Downloading package punkt to ../datasets/nltk/...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to ../datasets/nltk/...
[nltk_data]   Package stopwords is already up-to-date!


In [179]:
class Tokenizer:
    def __init__(self, csv_path: str, filter: str = "business"):
        self.documents = pl.read_csv(csv_path)
        self.documents = self._filter_by_label(filter)

        self.featurizer = TfidfVectorizer(stop_words = stopwords.words("english"),
                                          norm = "l1")

    def _filter_by_label(self, label: str | None):
        if not label:
            return self.documents
        return self.documents.filter( pl.col("labels") == label )

    def __getitem__(self, idx: int):
        print("#"*200)
        print(self.wrap(self.documents[idx]))
        print("#"*200)
        self.summerize(self.documents[idx])

    def wrap(self, x):
        return textwrap.fill(x["text"][0], replace_whitespace = False, fix_sentence_endings = True)

    def summerize(self, doc: str):
        _, text = self._split_title_text(doc)

        sents = sent_tokenize(text)
        x = self.featurizer.fit_transform(sents)

        scores = self.get_scores(x)
        sort_idx = np.argsort(-scores)

        self.generate_summary(sents, sort_idx)

    def _split_title_text(self, document: pl.Series) -> tuple[str, str]:
        return document["text"][0].split("\n", 1)

    def get_scores(self, rows ):
        non_zero_counts = (rows != 0).sum(axis=1).A1  # convert from matrix to 1D array
        row_sums = rows.sum(axis=1).A1

        scores = np.divide(row_sums, non_zero_counts, out=np.zeros_like(row_sums), where=non_zero_counts!=0)
        return scores

    def generate_summary(self, sents, indexes):
        for i in indexes[:5]:
            print(sents[int(i)])


In [180]:
Tokenizer("../datasets/bbc_text_cls.csv")[0]

########################################################################################################################################################################################################
Ad sales boost Time Warner profit

Quarterly profits at US media giant
TimeWarner jumped 76% to $1.13bn (£600m) for the three months to
December, from $639m year-earlier.

The firm, which is now one of the
biggest investors in Google, benefited from sales of high-speed
internet connections and higher advert sales.  TimeWarner said fourth
quarter sales rose 2% to $11.1bn from $10.9bn.  Its profits were
buoyed by one-off gains which offset a profit dip at Warner Bros, and
less users for AOL.

Time Warner said on Friday that it now owns 8% of
search-engine Google.  But its own internet business, AOL, had has
mixed fortunes.  It lost 464,000 subscribers in the fourth quarter
profits were lower than in the preceding three quarters.  However, the
company said AOL's underlying profit before exce